# ML-05 — Feature Vector and Leakage/Privacy Check

[w03_feature_leakage_check.ipynb](file:///c:/Users/Rida%20Eman/Downloads/Flyrank%20AI_intenship/work/notebooks/w03_feature_leakage_check.ipynb)

This notebook builds the feature vector for content opportunity scoring, audits every feature for leakage/privacy risks, and documents excluded columns.

## 1. Build the feature vector

We load the starter dataset, clean missing values, handle categorical columns, and create numerical features for training.

In [2]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Build feature vector
df['has_keyword_data'] = (~df['word_count'].isna()).astype(int)
df['word_count_clean'] = df['word_count'].fillna(0)
df['avg_position_clean'] = df['avg_position'].fillna(15.0)
df['ctr_clean'] = df['ctr'].fillna(0.0)
df['is_declining_target'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'sessions_90d', 'content_age_days', 'word_count_clean', 'avg_position_clean', 'ctr_clean', 'has_keyword_data']
print(f"Feature matrix shape: {df[feature_cols].shape}")
print(df[feature_cols].head())

Feature matrix shape: (30000, 7)
   impressions_90d  sessions_90d  ...  ctr_clean  has_keyword_data
0             3803            17  ...       0.76                 1
1            15320             9  ...       0.05                 1
2            12581            11  ...       0.09                 1
3            11751            78  ...       0.49                 0
4            19140           145  ...       0.13                 1

[5 rows x 7 columns]


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `impressions_90d`: 90-day Search Console impressions. Available at prediction time.
- `sessions_90d`: 90-day GA4 sessions. Available at prediction time.
- `content_age_days`: Days since content creation. Knowable from metadata.
- `word_count_clean`: Word count, filled 0 for missing. Knowable from article text.
- `avg_position_clean`: Average Google position, filled 15.0 for missing (pos 0 = no data).
- `ctr_clean`: Click-through rate percentage.
- `has_keyword_data`: Flag indicating whether keyword data is present.

In [4]:
for col in feature_cols:
    print(f"Feature '{col}': null count = {df[col].isna().sum()}, min = {df[col].min()}, max = {df[col].max()}")

Feature 'impressions_90d': null count = 0, min = 1, max = 517715
Feature 'sessions_90d': null count = 0, min = 1, max = 4345
Feature 'content_age_days': null count = 0, min = 90, max = 564
Feature 'word_count_clean': null count = 0, min = 0.0, max = 9546.0
Feature 'avg_position_clean': null count = 0, min = 0.0, max = 245.0
Feature 'ctr_clean': null count = 0, min = 0.0, max = 100.0
Feature 'has_keyword_data': null count = 0, min = 0, max = 1


## 3. The leakage hunt

We test for leakage by intentionally introducing target-derived columns (`trend_pct`) to observe artificial score inflation.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_safe = df[feature_cols]
y = df['is_declining_target']

# Leakage test
X_leaked = df[feature_cols + ['trend_pct']]

X_tr_s, X_te_s, y_tr, y_te = train_test_split(X_safe, y, test_size=0.25, random_state=42)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42)

clf_safe = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_s, y_tr)
print(f"Safe Model ROC-AUC: {roc_auc_score(y_te, clf_safe.predict_proba(X_te_s)[:, 1]):.4f}")

clf_leak = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_l, y_tr)
print(f"Leaked Model (with trend_pct) ROC-AUC: {roc_auc_score(y_te, clf_leak.predict_proba(X_te_l)[:, 1]):.4f}")

Safe Model ROC-AUC: 0.7400
Leaked Model (with trend_pct) ROC-AUC: 1.0000


## 4. What I excluded and why

- `trend_direction` / `trend_pct`: Excluded because the target label `is_declining_target` is directly derived from them (outcome leakage).
- `client_id` / `content_id`: Excluded as direct features to prevent memorization of specific client pseudonyms.
- `health_score` / `priority_score`: Excluded because they represent live product rule outputs.

In [8]:
excluded_list = ['trend_direction', 'trend_pct', 'client_id', 'content_id']
print(f"Excluded {len(excluded_list)} columns from feature vector.")

Excluded 4 columns from feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.